# 2 · Inference / Experiment Design

Turn an **idea** into the **best settings**. You define a search space + objective; the engine
sweeps it (with a progress bar), ranks the trials, and writes a documented results folder.
Then you interpret the table and pick a *robust* winner. See `docs/EXPERIMENT_GUIDE.md` for the method.


## 0 · Setup


In [ ]:
import sys, os, json, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..'))
    import quant
from quant.engine import BacktestConfig
from quant.strategies import EmaRibbon
from quant.viz import sweep_heatmap
from experiments.base import Experiment
print('ready')

## 1 · State the idea, then define the experiment
*Idea: for an EMA-ribbon entry, which EMA lengths + stop/target maximize Sharpe?*

- `strategy_space` varies strategy fields (EMA periods, confirm candles, `session`, `hours`).
- `cfg_space` varies execution config (stop-loss, take-profit, trailing, leverage).
- `min_trades` guards against a lucky 2-trade 'winner'.


In [ ]:
base_cfg = BacktestConfig(
    initial_cash=10_000, fee_bps=8, slippage_bps=1.5, spread=0.20,
    exit_enabled=True, sl_mode='entry_pct', tp_mode='rr',
    sizing_mode='risk_pct_equity', sizing_value=1.0, allow_rule_close=False,
)
exp = Experiment(
    name='ema_and_exit_search',
    description='Best EMA ribbon (fast/slow/confirm) x stop/target for gold, by Sharpe.',
    strategy_cls=EmaRibbon,
    base_cfg=base_cfg,
    symbol='PAXGUSDT', tf='1m', start='2026-01-01', end='2026-05-31',   # search window
    strategy_space={'fast': [20, 50, 100], 'slow': [150, 200], 'confirm_n': [1, 3, 5]},
    cfg_space={'sl_value': [0.4, 0.6, 1.0], 'tp_value': [1.5, 2.0, 3.0]},
    metric='sharpe', direction='max', min_trades=20,
    valid_fn=lambda p: p['fast'] < p['slow'],
)
n = (3*2*3) * (3*3)   # strategy combos x cfg combos
print('total trials:', n)

## 2 · Run it (progress bar)
The bar advances per strategy variant; each variant reuses its signals across all `cfg` combos
(fast). Results are also written to `experiments/results/<name>/` (results.csv, best.json, report.md).


In [ ]:
ranked = exp.run()          # tqdm progress bar; returns a ranked DataFrame
print('ranked trials:', len(ranked))

## 3 · Interpret the results
Ranked best-first by the objective. **Don't just take row 0** — look for a *plateau* of similar
scores (robust), check `num_trades`, and cross-check drawdown / profit factor.


In [ ]:
cols = ['fast','slow','confirm_n','sl_value','tp_value',
        'num_trades','total_return_pct','win_rate_pct','profit_factor','sharpe','max_drawdown_pct']
ranked[cols].head(12)

In [ ]:
# the winning settings (also saved to experiments/results/ema_and_exit_search/best.json)
best = ranked.iloc[0]
print('best sharpe: %.3f' % best['sharpe'])
print('strategy: fast=%d slow=%d confirm_n=%d' % (best['fast'], best['slow'], best['confirm_n']))
print('exit:     sl_value=%.2f  tp_value=%.2f' % (best['sl_value'], best['tp_value']))

### See the parameter surface (is the winner on a plateau?)
A robust setting has good *neighbours*; a lone bright spike is likely overfit.


In [ ]:
sweep_heatmap(ranked, x='fast', y='slow', z='sharpe')

## 4 · Pick a robust winner & VALIDATE out-of-sample
Re-run the chosen settings on a **different period** than you searched. If it holds up, trust it more.


In [ ]:
chosen = EmaRibbon(fast=int(best['fast']), slow=int(best['slow']), confirm_n=int(best['confirm_n']))
import dataclasses
chosen_cfg = dataclasses.replace(base_cfg, sl_value=float(best['sl_value']), tp_value=float(best['tp_value']))

from quant.data import get_ohlcv
oos = get_ohlcv('PAXGUSDT', '1m', start='2025-09-01', end='2025-12-31', tz='UTC')   # earlier, unseen
res_oos = chosen.backtest(oos, chosen_cfg)
print('OUT-OF-SAMPLE  sharpe=%.3f  return=%.2f%%  trades=%d  maxDD=%.2f%%' % (
    res_oos.stats['sharpe'], res_oos.stats['total_return_pct'],
    res_oos.stats['num_trades'], res_oos.stats['max_drawdown_pct']))

## 5 · How to read & choose (rules of thumb)
- **Plateau > peak.** Prefer a setting whose neighbours also score well.
- **Enough trades.** A great metric on <20 trades is noise (that's what `min_trades` filters).
- **Cross-check.** High Sharpe but tiny profit factor or huge drawdown = fragile.
- **Out-of-sample.** If it collapses on unseen data, it was curve-fit — discard it.
- **Costs on.** Keep realistic `spread`/`commission`; an edge that needs zero cost isn't real.
- **Fewer knobs.** Every tuned parameter is a chance to overfit.

Full method + anti-overfitting checklist: `docs/EXPERIMENT_GUIDE.md`. Shipped examples:
`python -m experiments.ema_mtf` · `experiments.session_timing` · `experiments.exit_design`.
